# Step 1: Train IQL (Baseline 2Q + Ensemble 3Q)

**CMPE 260 — Group 6**

This notebook trains IQL with both DoubleCritic (2Q) and TripleCritic (3Q) on all 3 D4RL environments.

**Runtime:** ~20 min per config on Colab T4 GPU (2 hours total for 6 runs)

**Outputs:**
- `results/training_{env}_{nQ}Q.csv` — eval scores during training
- `tmp/{env}_{nQ}Q/checkpoint_*/` — model checkpoints

In [ ]:
# Clone repo and install deps
!git clone -b main https://github.com/shloakk/iql-robustness-analysis.git
%cd iql-robustness-analysis

!pip install -q jax jaxlib flax optax
!pip install -q mujoco "gymnasium[mujoco]" gym
!pip install -q h5py tqdm matplotlib numpy scipy
!pip install -q absl-py ml_collections tensorboardX tensorflow-probability
!pip install -q git+https://github.com/Farama-Foundation/d4rl@master

In [ ]:
import jax
print(f"JAX devices: {jax.devices()}")
print(f"GPU available: {len(jax.devices('gpu')) > 0}")

In [ ]:
# Configuration
ENVIRONMENTS = [
    'hopper-medium-v2',
    'halfcheetah-medium-v2',
    'walker2d-medium-v2',
]
CRITIC_CONFIGS = [2, 3]  # 2Q (DoubleCritic) and 3Q (TripleCritic)
MAX_STEPS = 300_000
SEED = 42

In [ ]:
import os, sys, csv
import numpy as np
from tqdm import tqdm

sys.path.insert(0, '.')
os.environ['D4RL_SUPPRESS_IMPORT_ERROR'] = '1'

import gymnasium as gym
import wrappers
from iql import Learner
from iql.dataset_utils import D4RLDataset, split_into_trajectories, d4rl_to_gymnasium_name
from evaluation import evaluate

def normalize(dataset):
    trajs = split_into_trajectories(
        dataset.observations, dataset.actions, dataset.rewards,
        dataset.masks, dataset.dones_float, dataset.next_observations)
    def compute_returns(traj):
        return sum(rew for _, _, rew, _, _, _ in traj)
    trajs.sort(key=compute_returns)
    ret_range = compute_returns(trajs[-1]) - compute_returns(trajs[0])
    if abs(ret_range) < 1e-8:
        print(f'WARNING: reward range is ~0, skipping normalization')
    else:
        dataset.rewards /= ret_range
        dataset.rewards *= 1000.0

def make_env_and_dataset(env_name, seed):
    gym_env_name = d4rl_to_gymnasium_name(env_name)
    env = gym.make(gym_env_name)
    env = wrappers.EpisodeMonitor(env)
    env = wrappers.SinglePrecision(env)
    env.reset(seed=seed)
    env.action_space.seed(seed)
    env.observation_space.seed(seed)
    dataset = D4RLDataset(env_name)
    normalize(dataset)
    return env, dataset

CONFIG = {
    'actor_lr': 3e-4, 'value_lr': 3e-4, 'critic_lr': 3e-4,
    'hidden_dims': (256, 256), 'discount': 0.99,
    'expectile': 0.7, 'temperature': 3.0,
    'dropout_rate': None, 'tau': 0.005,
}

print('Setup complete.')

In [ ]:
os.makedirs('results', exist_ok=True)

for env_name in ENVIRONMENTS:
    for num_critics in CRITIC_CONFIGS:
        label = f'{num_critics}Q'
        critic_type = 'DoubleCritic' if num_critics == 2 else 'TripleCritic'
        print(f"\n{'='*60}")
        print(f"Training IQL ({label} {critic_type}) on {env_name}")
        print(f"{'='*60}")

        env, dataset = make_env_and_dataset(env_name, SEED)
        agent = Learner(
            SEED, env.observation_space.sample()[np.newaxis],
            env.action_space.sample()[np.newaxis],
            max_steps=MAX_STEPS, num_critics=num_critics, **CONFIG)

        eval_results = []
        for i in tqdm(range(1, MAX_STEPS + 1), desc=f'{env_name} {label}'):
            batch = dataset.sample(256)
            agent.update(batch)

            if i % 50_000 == 0:
                stats = evaluate(agent, env, 10)
                eval_results.append({'step': i, 'raw_return': stats['return'],
                                     'normalized_score': stats['return']})
                print(f"  Step {i:>7,}: score={stats['return']:.2f}")

        # Save results
        outfile = f'results/training_{env_name}_{label}.csv'
        with open(outfile, 'w', newline='') as f:
            w = csv.DictWriter(f, fieldnames=['step', 'raw_return', 'normalized_score'])
            w.writeheader()
            w.writerows(eval_results)
        print(f"  Saved: {outfile}")

        # Save checkpoint
        save_dir = f'tmp/{env_name}_{label}'
        agent.save(save_dir, MAX_STEPS)
        print(f"  Checkpoint: {save_dir}/checkpoint_{MAX_STEPS}/")
        env.close()

print("\n\u2705 All training complete (2Q + 3Q)!")

In [ ]:
# Download results
from google.colab import files
for env_name in ENVIRONMENTS:
    for num_critics in CRITIC_CONFIGS:
        files.download(f'results/training_{env_name}_{num_critics}Q.csv')